# JEDI3sub Simulation

This notebook is only for the trace simulator. It samples a 10 second real DS01 JEDI3sub recording when the local data folder is present, simulates a 10 second trace, and compares the visible trace, event-triggered waveform, distribution, and spectrum.

In [1]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from vnoiser import Jedi3SubConfig, JediSub3Dataset, simulate_jedi3sub_trace


In [2]:
duration_s = 10.0
seed = 12
dataset_root = next(
    (path for path in (Path("DS01-JEDI-sub3"), Path("../DS01-JEDI-sub3")) if path.exists()),
    None,
)

real = None
if dataset_root is not None:
    dataset = JediSub3Dataset(dataset_root)
    for offset in range(40):
        candidate = dataset.sample(duration_s=duration_s, seed=seed + offset, require_events=True)
        if len(candidate.events_ap_indices):
            real = candidate
            break

sim_config = Jedi3SubConfig()
sim = simulate_jedi3sub_trace(duration_s, config=sim_config, seed=seed)

print(f"sim: {len(sim.t):,} samples at {sim.metadata['fs_hz']:.0f} Hz | APs {len(sim.events['ap_indices'])} | fast {len(sim.events['fast_depol_indices'])} | slow {len(sim.events['slow_depol_indices'])} | bursts {len(sim.events['burst_indices'])}")
if real is not None:
    print(f"real: {real.path.name} | {len(real.t):,} samples at {real.fs_hz:.0f} Hz | APs {len(real.events_ap_indices)}")
else:
    print("real: DS01-JEDI-sub3 not found; showing simulation only")


sim: 100,000 samples at 10000 Hz | APs 90 | fast 211 | slow 15 | bursts 7
real: VAttached_stan60_expt1_scan42_apical2_mini.mat | 100,000 samples at 10000 Hz | APs 133


In [3]:
def robust_mad(y):
    y = np.asarray(y)
    return float(np.median(np.abs(y - np.median(y))) * 1.4826)

def event_average(trace, events, fs_hz, pre_ms=8.0, post_ms=20.0):
    trace = np.asarray(trace)
    events = np.asarray(events, dtype=int)
    pre = int(round(pre_ms * fs_hz / 1000.0))
    post = int(round(post_ms * fs_hz / 1000.0))
    keep = events[(events >= pre) & (events < len(trace) - post)]
    x = np.arange(-pre, post + 1) / fs_hz * 1000.0
    if len(keep) == 0:
        return x, np.full_like(x, np.nan, dtype=float), np.full_like(x, np.nan, dtype=float), np.full_like(x, np.nan, dtype=float)
    snippets = np.vstack([trace[i - pre : i + post + 1] for i in keep[:600]])
    baseline = snippets[:, :pre].mean(axis=1, keepdims=True)
    snippets = snippets - baseline
    return x, snippets.mean(axis=0), np.percentile(snippets, 25, axis=0), np.percentile(snippets, 75, axis=0)

def trace_stats(name, t, trace, events):
    fs_hz = 1.0 / np.median(np.diff(t))
    x, avg, _, _ = event_average(trace, events, fs_hz)
    trough_i = int(np.nanargmin(avg)) if np.isfinite(avg).any() else 0
    zero_i = int(np.argmin(np.abs(x)))
    return {
        "trace": name,
        "fs_hz": fs_hz,
        "std": float(np.std(trace)),
        "mad": robust_mad(trace),
        "p01": float(np.percentile(trace, 1)),
        "p99": float(np.percentile(trace, 99)),
        "diff_sd": float(np.std(np.diff(trace))),
        "ap_rate_hz": len(events) / (t[-1] - t[0]),
        "ap_pos0": float(avg[zero_i]) if np.isfinite(avg[zero_i]) else np.nan,
        "trough": float(avg[trough_i]) if np.isfinite(avg[trough_i]) else np.nan,
        "trough_ms": float(x[trough_i]) if np.isfinite(avg[trough_i]) else np.nan,
    }

def spectrum(t, y):
    fs_hz = 1.0 / np.median(np.diff(t))
    y = y - np.mean(y)
    n = min(len(y), int(fs_hz * 10))
    freq = np.fft.rfftfreq(n, d=1.0 / fs_hz)
    power = np.abs(np.fft.rfft(y[:n] * np.hanning(n))) ** 2
    keep = (freq >= 1) & (freq <= 2000)
    return freq[keep], power[keep] / np.max(power[keep])

stats = [trace_stats("sim", sim.t, sim.dff_observed, sim.events["ap_indices"])]
if real is not None:
    stats.insert(0, trace_stats("real", real.t, real.trace, real.events_ap_indices))
stats


[{'trace': 'real',
  'fs_hz': np.float64(10000.000000023307),
  'std': 0.4018693350624551,
  'mad': 0.21904011607506835,
  'p01': -0.5141179045301544,
  'p99': 2.0183530607019793,
  'diff_sd': 0.031045995321851535,
  'ap_rate_hz': np.float64(13.300133001330012),
  'ap_pos0': 1.3775532139450979,
  'trough': -0.8388556354221329,
  'trough_ms': -7.999999999981355},
 {'trace': 'sim',
  'fs_hz': np.float64(10000.000000023307),
  'std': 0.3932725195805324,
  'mad': 0.20967242123771163,
  'p01': -0.5460552904439836,
  'p99': 1.9444721670940444,
  'diff_sd': 0.04166882884142159,
  'ap_rate_hz': np.float64(9.000090000900009),
  'ap_pos0': 1.6655763846900615,
  'trough': -1.052950834510708,
  'trough_ms': 4.999999999988346}]

In [4]:
fig = make_subplots(
    rows=3,
    cols=2,
    specs=[[{}, {}], [{}, {}], [{}, {"type": "table"}]],
    subplot_titles=(
        "real 10 s reference",
        "simulated 10 s trace",
        "AP-triggered waveform",
        "sample distribution",
        "normalized spectrum",
        "summary metrics",
    ),
    horizontal_spacing=0.08,
    vertical_spacing=0.10,
)

if real is not None:
    fig.add_trace(go.Scatter(x=real.t, y=real.trace, mode="lines", name="real", line={"color": "#2f4858", "width": 1}), row=1, col=1)
    fig.add_trace(go.Scatter(x=real.events_ap_times_s, y=real.trace[real.events_ap_indices], mode="markers", name="real AP", marker={"color": "#d7263d", "size": 5}), row=1, col=1)
    x, avg, lo, hi = event_average(real.trace, real.events_ap_indices, real.fs_hz)
    fig.add_trace(go.Scatter(x=np.r_[x, x[::-1]], y=np.r_[hi, lo[::-1]], fill="toself", mode="lines", line={"width": 0}, name="real IQR", fillcolor="rgba(47,72,88,0.18)"), row=2, col=1)
    fig.add_trace(go.Scatter(x=x, y=avg, mode="lines", name="real AP avg", line={"color": "#2f4858", "width": 2}), row=2, col=1)
    fig.add_trace(go.Histogram(x=real.trace, histnorm="probability density", nbinsx=160, name="real", marker={"color": "rgba(47,72,88,0.55)"}), row=2, col=2)
    f, p = spectrum(real.t, real.trace)
    fig.add_trace(go.Scatter(x=f, y=p, mode="lines", name="real spectrum", line={"color": "#2f4858"}), row=3, col=1)

fig.add_trace(go.Scatter(x=sim.t, y=sim.dff_observed, mode="lines", name="sim observed", line={"color": "#4c78a8", "width": 1}), row=1, col=2)
fig.add_trace(go.Scatter(x=sim.events["ap_times_s"], y=sim.dff_observed[sim.events["ap_indices"]], mode="markers", name="sim AP", marker={"color": "#d7263d", "size": 5}), row=1, col=2)
fig.add_trace(go.Scatter(x=sim.events["fast_depol_times_s"], y=sim.dff_observed[sim.events["fast_depol_indices"]], mode="markers", name="fast depol", marker={"color": "#1b998b", "size": 4, "symbol": "line-ns-open"}), row=1, col=2)
fig.add_trace(go.Scatter(x=sim.events["slow_depol_times_s"], y=sim.dff_observed[sim.events["slow_depol_indices"]], mode="markers", name="slow depol", marker={"color": "#bc5090", "size": 7, "symbol": "diamond-open"}), row=1, col=2)
fig.add_trace(go.Scatter(x=sim.events["burst_times_s"], y=sim.dff_observed[sim.events["burst_indices"]], mode="markers", name="burst", marker={"color": "#ffa600", "size": 8, "symbol": "star"}), row=1, col=2)
x, avg, lo, hi = event_average(sim.dff_observed, sim.events["ap_indices"], sim.metadata["fs_hz"])
fig.add_trace(go.Scatter(x=np.r_[x, x[::-1]], y=np.r_[hi, lo[::-1]], fill="toself", mode="lines", line={"width": 0}, name="sim IQR", fillcolor="rgba(76,120,168,0.18)"), row=2, col=1)
fig.add_trace(go.Scatter(x=x, y=avg, mode="lines", name="sim AP avg", line={"color": "#4c78a8", "width": 2}), row=2, col=1)
fig.add_trace(go.Histogram(x=sim.dff_observed, histnorm="probability density", nbinsx=160, name="sim", marker={"color": "rgba(76,120,168,0.55)"}), row=2, col=2)
f, p = spectrum(sim.t, sim.dff_observed)
fig.add_trace(go.Scatter(x=f, y=p, mode="lines", name="sim spectrum", line={"color": "#4c78a8"}), row=3, col=1)

metric_names = ["fs_hz", "std", "mad", "p01", "p99", "diff_sd", "ap_rate_hz", "ap_pos0", "trough", "trough_ms"]
fig.add_trace(go.Table(
    header={"values": ["metric"] + [row["trace"] for row in stats], "fill_color": "#e8edf3", "align": "left"},
    cells={"values": [[m for m in metric_names]] + [[f"{row[m]:.3f}" for m in metric_names] for row in stats], "align": "left"},
), row=3, col=2)

fig.update_xaxes(title_text="time (s)", rangeslider={"visible": True}, row=1, col=1)
fig.update_xaxes(title_text="time (s)", rangeslider={"visible": True}, row=1, col=2)
fig.update_xaxes(title_text="time from AP (ms)", row=2, col=1)
fig.update_xaxes(title_text="trace value", row=2, col=2)
fig.update_xaxes(title_text="frequency (Hz)", type="log", row=3, col=1)
fig.update_yaxes(title_text="trace", row=1, col=1)
fig.update_yaxes(title_text="trace", row=1, col=2)
fig.update_yaxes(title_text="event average", row=2, col=1)
fig.update_yaxes(title_text="density", row=2, col=2)
fig.update_yaxes(title_text="normalized power", type="log", row=3, col=1)
fig.update_layout(height=920, template="plotly_white", hovermode="x unified", barmode="overlay", legend={"orientation": "h", "y": 1.04})
fig.show()
